In [105]:
import pandas as pd
import numpy as np
from pathlib import Path
import geopandas as gpd
import sys

# ---------------------------------------------------------------------
# Add project root to Python path
# ---------------------------------------------------------------------


# project root = two levels above notebooks
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

In [106]:
from src.data.data_loading import (
    load_full_dataset,
    load_full_training_pool,
    load_test_data,
    load_training_data,
    load_validation_data
)
import xgboost as xgb

In [107]:
train_df = load_training_data(cv_fold=0)
val_df = load_validation_data(cv_fold=0)

In [108]:
print(f"Length of train: {len(train_df)}")
print(f"Length of train: {len(val_df)}")
print(f"Unique FSAs in train: {train_df['FSA'].nunique()}")
print(f"Unique FSAs in val: {val_df['FSA'].nunique()}")

Length of train: 8392
Length of train: 2069
Unique FSAs in train: 663
Unique FSAs in val: 191


In [109]:
removed_columns = ['spatial_cluster', 'is_test', 'cv_fold']
train_df = train_df.drop(columns= removed_columns)
val_df = val_df.drop(columns=removed_columns)

In [110]:
train_df['y_binary'] = (train_df['concentration'] >200).astype(int)
val_df['y_binary'] = (val_df['concentration'] > 200).astype(int)

In [ ]:
def fsa_features_groupby(df):
    fsa_features = df.drop(columns="y_binary").drop_duplicates("FSA")
    y_mean = df.groupby("FSA")["y_binary"].mean().reset_index(name="y_mean")
    return fsa_features.merge(y_mean, on="FSA")

def validation_X_y(df):
    fsa_features = fsa_features_groupby(df)
    X_val = fsa_features.drop(
        columns=['concentration', 'provinceterritory', 'FSA', 'geometry', 'y_mean']
        )
    y_val = (fsa_features['y_mean'] > 0.5).astype(int)
    return (X_val,y_val)

def X_train(df, approach = "Naive"):
    if approach == "Naive":
        return df.drop(columns=['concentration', 'y_binary', 'provinceterritory', 'FSA', 'geometry'])
    ## Later when we try sophisticated approaches
    elif approach == "TBD":
        columns_to_drop = ['concentration', 'y_binary', 'provinceterritory', 'FSA', 'geometry']
        return df.drop(columns= columns_to_drop)


In [ ]:
pos_count = train_df['y_binary'].sum()
neg_count = len(train_df) - pos_count

clf = xgb.XGBClassifier(
    objective="binary:logistic",
    tree_method="hist",
    eval_metric="aucpr",
    scale_pos_weight=neg_count / pos_count,
    max_depth=5,
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
clf.fit(
    X_train(train_df), 
    train_df['y_binary'],
    eval_set = [validation_X_y(val_df)],
    verbose = True
    )

[0]	validation_0-aucpr:0.03045
[1]	validation_0-aucpr:0.05177
[2]	validation_0-aucpr:0.05006
[3]	validation_0-aucpr:0.03777
[4]	validation_0-aucpr:0.04294
[5]	validation_0-aucpr:0.04220
[6]	validation_0-aucpr:0.04654
[7]	validation_0-aucpr:0.04665
[8]	validation_0-aucpr:0.04810
[9]	validation_0-aucpr:0.04297
[10]	validation_0-aucpr:0.04755
[11]	validation_0-aucpr:0.04887
[12]	validation_0-aucpr:0.05099
[13]	validation_0-aucpr:0.05010
[14]	validation_0-aucpr:0.05175
[15]	validation_0-aucpr:0.05503
[16]	validation_0-aucpr:0.05808
[17]	validation_0-aucpr:0.05715
[18]	validation_0-aucpr:0.05511
[19]	validation_0-aucpr:0.05252
[20]	validation_0-aucpr:0.05270
[21]	validation_0-aucpr:0.05502
[22]	validation_0-aucpr:0.05439
[23]	validation_0-aucpr:0.05563
[24]	validation_0-aucpr:0.05666
[25]	validation_0-aucpr:0.05818
[26]	validation_0-aucpr:0.05868
[27]	validation_0-aucpr:0.06030
[28]	validation_0-aucpr:0.06114
[29]	validation_0-aucpr:0.05775
[30]	validation_0-aucpr:0.05631
[31]	validation_0-

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [113]:
clf.predict_proba(validation_features_group_by_fsa(val_df))[:,1]

array([0.01497509, 0.07552649, 0.10289402, 0.01837405, 0.10116141,
       0.4732445 , 0.52239406, 0.12395569, 0.26530766, 0.3012088 ,
       0.06833049, 0.0154706 , 0.11661884, 0.05518072, 0.05446896,
       0.01983453, 0.3661882 , 0.14253925, 0.29033875, 0.22909458,
       0.16233572, 0.487902  , 0.70775247, 0.68880975, 0.4977461 ,
       0.4313172 , 0.08510341, 0.34122714, 0.36323628, 0.5955584 ,
       0.5983987 , 0.24317764, 0.5477125 , 0.22603467, 0.19967325,
       0.57842785, 0.14633529, 0.17231677, 0.23038709, 0.2976119 ,
       0.08655594, 0.3146301 , 0.27054578, 0.2690262 , 0.02958285,
       0.11833373, 0.21874851, 0.24264959, 0.14305985, 0.18801367,
       0.28204086, 0.15339121, 0.08369931, 0.02781712, 0.23999399,
       0.03041207, 0.14727063, 0.30233407, 0.15825443, 0.15959114,
       0.02580544, 0.23407109, 0.12927891, 0.16344605, 0.10707135,
       0.04019822, 0.14040458, 0.01695174, 0.10581995, 0.11304417,
       0.0927879 , 0.0552244 , 0.00563091, 0.03855657, 0.00440